# Script 3 — Treinamento dos Modelos ML
**Nested Cross-Validation | Ridge · SVR · Random Forest · Gradient Boosting**

In [ ]:

import pandas as pd, numpy as np, pickle, warnings, json
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, GridSearchCV, cross_validate
import joblib

warnings.filterwarnings('ignore')
PASTA_SAIDA = Path('outputs')
print("✅ Dependências carregadas")


In [ ]:

# Carrega artefatos do Script 2
treino   = pd.read_parquet(PASTA_SAIDA / 'treino.parquet')
teste    = pd.read_parquet(PASTA_SAIDA / 'teste.parquet')
with open(PASTA_SAIDA / 'features.pkl','rb') as f: FEATURES = pickle.load(f)
with open(PASTA_SAIDA / 'targets.pkl','rb') as f:  TARGETS  = pickle.load(f)
print(f"Treino: {treino.shape} | Teste: {teste.shape}")
print(f"Features: {len(FEATURES)} | Targets: {TARGETS}")


## Definição dos 4 algoritmos + grades de hiperparâmetros

In [ ]:

# ── K-Folds ────────────────────────────────────────────────────────────────
kf_externo = KFold(n_splits=5, shuffle=True, random_state=42)
kf_interno = KFold(n_splits=5, shuffle=True, random_state=43)

# ── Algoritmo 1: Ridge regularizado ───────────────────────────────────────
estimador_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge",  Ridge()),
])
grade_ridge = {
    "ridge__alpha": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
}

# ── Algoritmo 2: SVR kernel RBF ───────────────────────────────────────────
estimador_svr = Pipeline([
    ("scaler", StandardScaler()),
    ("svr",    SVR(kernel="rbf", max_iter=10000)),
])
grade_svr = {
    "svr__C":       [0.1, 1.0, 10.0, 100.0],
    "svr__epsilon": [0.01, 0.05, 0.1, 0.5],
    "svr__gamma":   ["scale", "auto"],
}

# ── Algoritmo 3: Random Forest ────────────────────────────────────────────
estimador_rf = RandomForestRegressor(
    n_estimators=300, random_state=42, n_jobs=-1
)
grade_rf = {
    "max_depth":        [None, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5],
    "max_features":     ["sqrt", "log2"],
}

# ── Algoritmo 4: Gradient Boosting ───────────────────────────────────────
estimador_gb = GradientBoostingRegressor(random_state=42)
grade_gb = {
    "n_estimators":  [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth":     [3, 5],
    "subsample":     [0.7, 0.8, 1.0],
}

ALGORITMOS = {
    "Ridge":            (estimador_ridge, grade_ridge),
    "SVR":              (estimador_svr,   grade_svr),
    "RandomForest":     (estimador_rf,    grade_rf),
    "GradientBoosting": (estimador_gb,    grade_gb),
}
print(f"✅ {len(ALGORITMOS)} algoritmos configurados")


## Nested Cross-Validation + Treinamento

In [ ]:

def treinar_alg(nome, estimador, grade, X, y, kf_int, kf_ext):
    """Nested CV: loop interno = seleção de hiperparâmetros, externo = avaliação."""
    # Loop interno
    gs = GridSearchCV(
        estimador, grade,
        cv=kf_int,
        scoring='neg_mean_squared_error',
        refit=True, n_jobs=-1,
    )
    gs.fit(X, y)
    melhor = gs.best_estimator_

    # Loop externo (estimativa não enviesada)
    cv_res = cross_validate(
        melhor, X, y, cv=kf_ext,
        scoring=['neg_mean_squared_error','neg_mean_absolute_error','r2'],
        return_train_score=False,
    )
    rmse = np.sqrt(-cv_res['test_neg_mean_squared_error'].mean())
    mae  = -cv_res['test_neg_mean_absolute_error'].mean()
    r2   = cv_res['test_r2'].mean()
    mape_vals = []
    for train_idx, val_idx in kf_ext.split(X):
        Xv, yv = X[val_idx], y[val_idx]
        yp = melhor.predict(Xv)
        mask = yv != 0
        if mask.sum() > 0:
            mape_vals.append(np.mean(np.abs((yv[mask]-yp[mask])/yv[mask])))
    mape = np.mean(mape_vals) if mape_vals else np.nan

    print(f"  {nome:<20} RMSE={rmse:,.0f}  MAE={mae:,.0f}  MAPE={mape:.2%}  R²={r2:.3f}")
    print(f"    Melhores params: {gs.best_params_}")
    return melhor, {'RMSE_CV':rmse,'MAE_CV':mae,'MAPE_CV':mape,'R2_CV':r2,
                    'best_params':gs.best_params_}

resultados = {}   # {target: {algoritmo: (modelo, metricas)}}

for target in TARGETS:
    print(f"\n{'='*60}")
    print(f"TARGET: {target}")
    print('='*60)
    df_t = treino[FEATURES + [target]].dropna()
    X = df_t[FEATURES].values
    y = df_t[target].values
    resultados[target] = {}
    for nome, (est, grade) in ALGORITMOS.items():
        modelo, metricas = treinar_alg(nome, est, grade, X, y, kf_interno, kf_externo)
        resultados[target][nome] = (modelo, metricas)
        joblib.dump(modelo, PASTA_SAIDA / f'modelo_{target}_{nome}.pkl')


## Resumo dos resultados de CV

In [ ]:

rows = []
for target, algs in resultados.items():
    for alg, (_, metricas) in algs.items():
        rows.append({'Target': target, 'Algoritmo': alg, **metricas})

df_res = pd.DataFrame(rows).drop(columns=['best_params'])
print(df_res.to_string(index=False))

df_res.to_csv(PASTA_SAIDA / 'resultados_cv.csv', index=False)
with open(PASTA_SAIDA / 'resultados_cv.pkl', 'wb') as f:
    pickle.dump(resultados, f)
print("\n✅ Modelos e resultados salvos")
